<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML321ENSkillsNetwork817-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Regression-based Rating Score Prediction using Embedding Features**


Estimated time needed: **45** minutes


In our previous lab, you have trained a neural network to predict the user-item interactions while simultaneously extracting the user and item embedding features. In the neural network, extends this by using  two embedding vectors as an input into a Neural Network to predict the rating.


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/module_4/images/rating_regression.png)



Another way to make rating predictions is to use the embedding as an input to a neural network by aggregating them into a single feature vector as input data `X`. 

With the interaction label `Y` such as a rating score or an enrollment mode, we can build our other standalone predictive models to approximate the mapping from `X` to `Y`, as shown in the above flowchart.


In this lab, you will be given the course interaction feature vectors as input data `X` and consider label `Y` as the numerical rating scores. As such, we turn the recommender system into a common regression task and you can apply what you have learned about regression modeling to predict the ratings.


## Objectives


After completing this lab you will be able to:


* Build regression models to predict ratings using the combined embedding vectors


----


## Prepare and setup lab environment


First install and import required libraries:


In [1]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn import linear_model
from sklearn.metrics import mean_squared_error

In [3]:
# also set a random state
rs = 123

In [4]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn import linear_model
from sklearn.metrics import mean_squared_error

### Load datasets


In [5]:
rating_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMSkillsNetwork-ML0321EN-Coursera/labs/v2/module_3/ratings.csv"
# Embeddings produced by lab_jupyter_cf_ann.ipynb (produce-your-own route) -- same deviation as cf_classification_w_embeddings.ipynb
user_emb_url = "../data/user_embeddings.csv"
item_emb_url = "../data/course_embeddings.csv"

The first dataset is the rating dataset that contains a user-item interaction matrix


In [6]:
rating_df = pd.read_csv(rating_url)

In [7]:
rating_df.head()

,user,item,rating
0,1889878,CC0101EN,5
1,1342067,CL0101EN,3
2,1990814,ML0120ENv3,5
3,380098,BD0211EN,5
4,779563,DS0101EN,3


As you can see from the above data, the user and item are just ids, let's substitute them by their embedding vectors:


In [8]:
# Load user embeddings
user_emb = pd.read_csv(user_emb_url)
# Load item embeddings
item_emb = pd.read_csv(item_emb_url)

In [9]:
user_emb.head()

,user,UFeature0,UFeature1,UFeature2,UFeature3,UFeature4,UFeature5,UFeature6,UFeature7,UFeature8,UFeature9,UFeature10,UFeature11,UFeature12,UFeature13,UFeature14,UFeature15
0,1889878,-0.169329,-0.171417,0.077784,-0.095310,0.182348,-0.077800,-0.087430,0.056138,-0.131548,0.036847,-0.063690,0.013257,-0.125162,0.068581,0.057402,-0.175451
1,1342067,-0.101114,0.000926,0.023026,0.002143,0.077819,-0.076868,-0.087827,-0.071640,-0.071966,0.015049,0.007795,-0.019745,-0.089747,0.115308,0.035302,0.091108
2,1990814,0.048882,0.242381,0.142094,-0.023209,0.214116,0.083913,-0.126982,-0.125498,0.016538,-0.199776,0.065957,-0.011468,-0.025896,-0.022052,-0.062476,-0.001740
3,380098,0.125124,0.091391,-0.064431,0.006099,0.121973,-0.050488,-0.122787,0.053204,-0.057795,0.040821,-0.026236,-0.002356,-0.109218,-0.059387,0.149408,0.113286
4,779563,0.061062,-0.127827,-0.072394,-0.026350,-0.066526,-0.011174,0.042442,-0.096202,0.134940,0.101634,0.006678,-0.024174,0.067476,-0.179815,0.072761,0.133266


In [10]:
item_emb.head()

,item,CFeature0,CFeature1,CFeature2,CFeature3,CFeature4,CFeature5,CFeature6,CFeature7,CFeature8,CFeature9,CFeature10,CFeature11,CFeature12,CFeature13,CFeature14,CFeature15
0,CC0101EN,-0.008491,0.009244,0.001581,0.026333,0.008334,0.019389,0.003154,-0.035585,-0.006947,-0.016610,0.033818,0.007301,0.016022,-0.013650,0.000223,-0.001176
1,CL0101EN,-0.029734,0.004281,0.030478,-0.043487,-0.029202,-0.030779,0.046974,0.015391,0.038814,0.000829,-0.004464,-0.022572,0.055641,-0.029218,0.015049,0.011639
2,ML0120ENv3,-0.016919,-0.010686,-0.010309,-0.042168,0.001871,-0.002098,-0.039025,-0.043623,-0.034600,-0.055885,-0.019527,-0.029323,-0.010548,-0.023231,0.038712,0.001239
3,BD0211EN,0.017437,-0.008801,0.032791,0.000298,-0.011701,-0.000531,-0.007562,0.040755,-0.005233,0.003373,-0.027239,-0.009436,0.006211,0.001653,0.013483,0.026129
4,DS0101EN,0.003147,-0.030302,-0.015159,-0.002713,0.004832,0.027061,-0.001908,-0.004655,-0.008233,0.005730,0.005575,0.016058,-0.007969,0.005005,-0.002917,-0.002826


In [11]:
# Merge user embedding features
user_emb_merged = pd.merge(rating_df, user_emb, how='left', left_on='user', right_on='user').fillna(0)
# Merge course embedding features
merged_df = pd.merge(user_emb_merged, item_emb, how='left', left_on='item', right_on='item').fillna(0)

In [12]:
merged_df.head()

,user,item,rating,UFeature0,UFeature1,UFeature2,UFeature3,UFeature4,UFeature5,UFeature6,...,CFeature6,CFeature7,CFeature8,CFeature9,CFeature10,CFeature11,CFeature12,CFeature13,CFeature14,CFeature15
0,1889878,CC0101EN,5,-0.169329,-0.171417,0.077784,-0.095310,0.182348,-0.077800,-0.087430,...,0.003154,-0.035585,-0.006947,-0.016610,0.033818,0.007301,0.016022,-0.013650,0.000223,-0.001176
1,1342067,CL0101EN,3,-0.101114,0.000926,0.023026,0.002143,0.077819,-0.076868,-0.087827,...,0.046974,0.015391,0.038814,0.000829,-0.004464,-0.022572,0.055641,-0.029218,0.015049,0.011639
2,1990814,ML0120ENv3,5,0.048882,0.242381,0.142094,-0.023209,0.214116,0.083913,-0.126982,...,-0.039025,-0.043623,-0.034600,-0.055885,-0.019527,-0.029323,-0.010548,-0.023231,0.038712,0.001239
3,380098,BD0211EN,5,0.125124,0.091391,-0.064431,0.006099,0.121973,-0.050488,-0.122787,...,-0.007562,0.040755,-0.005233,0.003373,-0.027239,-0.009436,0.006211,0.001653,0.013483,0.026129
4,779563,DS0101EN,3,0.061062,-0.127827,-0.072394,-0.026350,-0.066526,-0.011174,0.042442,...,-0.001908,-0.004655,-0.008233,0.005730,0.005575,0.016058,-0.007969,0.005005,-0.002917,-0.002826


Next, we can combine the user features (the column labels starting with `UFeature` and item features (the column labels starting with `CFeature`. In machine learning, there are many ways to aggregate two feature vectors such as element-wise add, multiply, max/min, average, etc. Here we simply add the two sets of feature columns:


In [13]:
# Define column names for user and course embedding features
u_features = [f"UFeature{i}" for i in range(16)] # Assuming there are 16 user embedding features
c_features = [f"CFeature{i}" for i in range(16)]  # Assuming there are 16 course embedding features

# Extract user embedding features
user_embeddings = merged_df[u_features]
# Extract course embedding features
course_embeddings = merged_df[c_features]
# Extract ratings
ratings = merged_df['rating']

# Aggregate the two feature columns using element-wise add
regression_dataset = user_embeddings + course_embeddings.values
# Rename the columns of the resulting DataFrame
regression_dataset.columns = [f"Feature{i}" for i in range(16)]# Assuming there are 16 features
# Add the 'rating' column from the original DataFrame to the regression dataset
regression_dataset['rating'] = ratings
# Display the first few rows of the regression dataset
regression_dataset.head()

,Feature0,Feature1,Feature2,Feature3,Feature4,Feature5,Feature6,Feature7,Feature8,Feature9,Feature10,Feature11,Feature12,Feature13,Feature14,Feature15,rating
0,-0.177820,-0.162174,0.079365,-0.068978,0.190682,-0.058411,-0.084276,0.020553,-0.138495,0.020237,-0.029873,0.020558,-0.109140,0.054931,0.057625,-0.176627,5
1,-0.130848,0.005207,0.053504,-0.041344,0.048617,-0.107647,-0.040853,-0.056250,-0.033151,0.015878,0.003331,-0.042317,-0.034107,0.086089,0.050351,0.102747,3
2,0.031963,0.231696,0.131785,-0.065377,0.215987,0.081816,-0.166008,-0.169121,-0.018062,-0.255661,0.046429,-0.040791,-0.036444,-0.045283,-0.023765,-0.000501,5
3,0.142561,0.082590,-0.031639,0.006397,0.110272,-0.051019,-0.130349,0.093959,-0.063027,0.044194,-0.053474,-0.011792,-0.103006,-0.057734,0.162891,0.139415,5
4,0.064209,-0.158129,-0.087553,-0.029063,-0.061694,0.015887,0.040534,-0.100857,0.126708,0.107364,0.012253,-0.008116,0.059507,-0.174809,0.069844,0.130440,3


By now, we have built the input dataset `X` and the output vector `y`:


In [14]:
X = regression_dataset.iloc[:, :-1]
y = regression_dataset.iloc[:, -1]
print(f"Input data shape: {X.shape}, Output data shape: {y.shape}")

Input data shape: (233306, 16), Output data shape: (233306,)


## TASK: Perform regression on the interaction dataset


Now our input data `X` and output `y` are ready, let's build regression models to map X to y and predict ratings. 


y.unique()


You may use `sklearn` to train and evaluate various regression models.


_TODO: First split dataset into training and testing datasets_


In [15]:
# test_size=0.2 to mirror cf_classification_w_embeddings.ipynb's split (this notebook's own hint
# suggests 0.3, but matching cf_classification keeps the two directly comparable)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=rs)

<details>
    <summary>Click here for Hints</summary>
    
Use `train_test_split()` to split dataset into training and testing datasets.  Use `X, y` as input dataset and output vector. Don't forget to specify `random_state = rs` and `test_size=0.3`.


_TODO: Create a basic linear regression model_


In [16]:
model = linear_model.LinearRegression()

<details>
    <summary>Click here for Hints</summary>
    
You can call `linear_regression = LinearRegression()` method to create a Linear Regression object


_TODO: Train the basic regression model with training data_


In [17]:
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](16,)","[-0.03, 0.03,-0.06,..., 0. , 0.01, 0.07]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](16,)","['Feature0','Feature1','Feature2',...,'Feature13','Feature14','Feature15']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,3.995
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,16
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,16


<details>
    <summary>Click here for Hints</summary>
    
You can call `model.fit()` method with `X_train, y_train` parameters.


_TODO: Evaluate the basic regression model_


In [18]:
import numpy as np
from sklearn.metrics import mean_absolute_error

y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"Predicted rating range on test set: [{y_pred.min():.4f}, {y_pred.max():.4f}]")

RMSE: 0.8124
MAE: 0.6633
Predicted rating range on test set: [3.9428, 4.0549]


In [19]:
import os

# Predict-then-rank. Score every (user, unseen-course) pair via a single vectorized predict()
# call -- not a per-pair loop, which would be ~33,901 x 126 = 4.3M calls. SCORE = predicted
# rating directly (this is a regressor, not a classifier -- no probability-to-value mapping
# needed, unlike cf_classification_w_embeddings.ipynb).
user_ids = user_emb["user"].tolist()
course_ids = item_emb["item"].tolist()
user_matrix = user_emb[u_features].to_numpy(dtype="float32")
course_matrix = item_emb[c_features].to_numpy(dtype="float32")

num_users = len(user_ids)
num_courses = len(course_ids)

# Element-wise-add every user embedding with every course embedding (same aggregation as
# regression_dataset above), flattened to a single (num_users * num_courses, 16) batch
interaction_matrix = user_matrix[:, None, :] + course_matrix[None, :, :]
interaction_matrix = interaction_matrix.reshape(-1, interaction_matrix.shape[-1])

predicted_ratings = model.predict(interaction_matrix)
score_matrix = predicted_ratings.reshape(num_users, num_courses)

print(f"Predicted rating range over the full (user x unseen-course) grid: "
      f"[{score_matrix.min():.4f}, {score_matrix.max():.4f}]")

# Mask out courses each user has already rated, restricted to this model's 126-course universe
enrolled_pivot = rating_df.assign(_v=1).pivot_table(index="user", columns="item", values="_v", aggfunc="max")
enrolled_pivot = enrolled_pivot.reindex(index=user_ids, columns=course_ids, fill_value=0).fillna(0)
enrolled_mask = enrolled_pivot.to_numpy() > 0

score_matrix_masked = np.where(enrolled_mask, -np.inf, score_matrix)

TOP_N_PER_USER = 20
top_n_idx = np.argsort(-score_matrix_masked, axis=1)[:, :TOP_N_PER_USER]

rows = []
for u_idx, user_id in enumerate(user_ids):
    for c_idx in top_n_idx[u_idx]:
        score = score_matrix_masked[u_idx, c_idx]
        if np.isfinite(score):
            rows.append((user_id, course_ids[c_idx], score))

cf_regression_res_df = pd.DataFrame(rows, columns=["USER", "COURSE_ID", "SCORE"])

os.makedirs("../data/predictions", exist_ok=True)
cf_regression_res_df.to_csv("../data/predictions/cf_regression.csv", index=False)
print(f"Saved {cf_regression_res_df.shape[0]} rows to data/predictions/cf_regression.csv "
      f"({cf_regression_res_df['USER'].nunique()} users)")

D:\ibm ml capstone\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


Predicted rating range over the full (user x unseen-course) grid: [3.9336, 4.0626]


Saved 678020 rows to data/predictions/cf_regression.csv (33901 users)


<details>
    <summary>Click here for Hints</summary>
    
You can call `model.predict()` method with `X_test` parameter to get model predictions. Then use `mean_squared_error()` with `y_test, your_predictions` parameters to calculate the RMSE. 


_TODO: Try different regression models such as Ridge, Lasso, ElasticNet and tune their hyperparameters to see which one has the best performance_


In [20]:
### WRITE YOUR CODE HERE


### Summary


In this lab, you have built regression models to predict numerical course ratings using the embedding feature vectors extracted from neural networks. In the next lab, we can treat the prediction problem as a classification problem as rating only has two categorical values so classification can be a more natural problem statement.


## Authors


[Yan Luo](https://www.linkedin.com/in/yan-luo-96288783/)


### Other Contributors


```toggle## Change Log
```


```toggle|Date (YYYY-MM-DD)|Version|Changed By|Change Description|
```
```toggle|-|-|-|-|
```
```toggle|2021-10-25|1.0|Yan|Created the initial version|
```


Copyright © 2021 IBM Corporation. All rights reserved.
